In [1]:
execution_to_fix = "output_modeling_20241022_071500"

# Joining browken-down modules
This notebook joins the execution outputs where the "split largest hyperparameter" option was used, creating a standardised output.

## 1. Import libraries and packages

In [ ]:
!pip install pandas numpy openpyxl

In [2]:
## Import libraries

# Pandas
import pandas as pd

# Json
import json

# OS
import os

# Itertools
from itertools import product

## 2. Read data

In [3]:
# Read the config file
folder = "./05_outputs_modeling/" + execution_to_fix
config_file = folder + "/config_file.json"
with open(config_file, 'r') as file:
    config = json.load(file)

## 3. Check all executions are there

In [4]:
def strings_to_floats_or_strings(string_list):
    def convert_if_float(s):
        try:
            if "." in s:
                return float(s)
            else:
                return int(s)
        except ValueError:
            return s

    return [tuple(convert_if_float(part) for part in s.split('_')) for s in string_list]

In [5]:
# Get all hyperparameter list
lst_hyperparams = [config['model']['parameters'][hyperparam] for hyperparam in config['model']['split largest hyperparameter']]
lst_all_executions_expected = list(product(*lst_hyperparams))

In [6]:
# Get actual executions
lst_all_executions_real = strings_to_floats_or_strings(os.listdir(folder))

In [7]:
# Check all executions are there
if not(all(elem in lst_all_executions_real for elem in lst_all_executions_expected)):

    # Print executions that are not there
    missing_exec = [elem for elem in lst_all_executions_expected if elem not in lst_all_executions_real]

    # Raise exeption
    raise Exception("Not all executions have been run")

## 4. Fix files
### 4.1. Building summary

In [8]:
# Create an empty list to store the DataFrames
building_summaries = []

# Walk through the directory
for root, dirs, files in os.walk(folder):
    for file in files:
        # Check if the file name matches the pattern
        if file.startswith("Building summary_") and file.endswith(".csv"):
            # Construct the full file path
            file_path = os.path.join(root, file)
            # Read the CSV file and append to the list
            df = pd.read_csv(file_path)
            building_summaries.append(df)

# Concatenate all DataFrames into a single DataFrame
df_building_summary = pd.concat(building_summaries, ignore_index=True)

# Drop unnamed column
df_building_summary = df_building_summary.drop(columns=['Unnamed: 0'])

# Create index column
df_building_summary['index_col'] = df_building_summary.index
df_building_summary = df_building_summary[['index_col'] + [col for col in df_building_summary.columns if col != 'index_col']]

In [9]:
# Export file
df_building_summary.drop(columns=['index_col']).to_csv(folder + "/Building summary_" + execution_to_fix[-15:] + ".csv")

### 4.2. Dashboard output

In [10]:
# Initialize dictionary to save tabs
dict_dashboard_output = {}

First, the "General" tab.

In [11]:
# Create an empty list to store the DataFrames
general_tabs = []

# Walk through the directory
for root, dirs, files in os.walk(folder):
    for file in files:
        # Check if the file name matches the pattern
        if file.startswith("Dashboard output") and file.endswith(".xlsx"):
            # Construct the full file path
            file_path = os.path.join(root, file)
            # Read the CSV file and append to the list
            df = pd.read_excel(file_path, sheet_name="General")
            general_tabs.append(df)

# Concatenate all DataFrames into a single DataFrame
df_general_tabs = pd.concat(general_tabs, ignore_index=True)

In [12]:
# Get best row
min_metric_1_index = df_general_tabs['Metric 1 test'].idxmin()
min_metric_1_name = df_general_tabs['Execution name'][min_metric_1_index]

# Fix dataframe
df_general_tabs_fixed = pd.DataFrame({
    'Execution name': [execution_to_fix[-15:]],
    'Model type': [df_general_tabs['Model type'].iloc[0]],
    'PCA': [df_general_tabs['PCA'].iloc[0]],
    'Number of search iterations': [df_general_tabs['Number of search iterations'].sum()],
    'Number of folds': [df_general_tabs['Number of folds'].iloc[0]]
    'Train time': [df_general_tabs['Train time'].sum()],
    'Metric 1': [df_general_tabs['Metric 1'].iloc[0]],
    'Metric 1 train': [df_general_tabs['Metric 1 train'].iloc[min_metric_1_index]],
    'Metric 1 test': [df_general_tabs['Metric 1 test'].iloc[min_metric_1_index]],
    'Metric 2': [df_general_tabs['Metric 2'].iloc[0]],
    'Metric 2 train': [df_general_tabs['Metric 2 train'].iloc[min_metric_1_index]],
    'Metric 2 test': [df_general_tabs['Metric 2 test'].iloc[min_metric_1_index]]
})

In [13]:
dict_dashboard_output['General'] = df_general_tabs_fixed.copy()

Now the "Train predictions" tab

In [14]:
# Create an empty list to store the DataFrames
train_preds_tabs = []

# Walk through the directory
for root, dirs, files in os.walk(folder):
    for file in files:
        # Check if the file name matches the pattern
        if file.startswith("Dashboard output_" + min_metric_1_name):
            # Construct the full file path
            file_path = os.path.join(root, file)
            # Read the CSV file and append to the list
            df = pd.read_excel(file_path, sheet_name="Train predictions")
            # Modify column names
            df.columns = [col if not str(col).startswith("Real") else "Real_" + execution_to_fix[-15:] for col in df.columns]
            # Save
            train_preds_tabs.append(df)

# Concatenate all DataFrames into a single DataFrame
df_train_preds_tabs = pd.concat(train_preds_tabs, ignore_index=True)

In [15]:
dict_dashboard_output['Train predictions'] = df_train_preds_tabs.copy()

Now the "Test predictions" tab

In [16]:
# Create an empty list to store the DataFrames
test_preds_tabs = []

# Walk through the directory
for root, dirs, files in os.walk(folder):
    for file in files:
        # Check if the file name matches the pattern
        if file.startswith("Dashboard output_" + min_metric_1_name):
            # Construct the full file path
            file_path = os.path.join(root, file)
            # Read the CSV file and append to the list
            df = pd.read_excel(file_path, sheet_name="Test predictions")
            # Modify column names
            df.columns = [col if not str(col).startswith("Real") else "Real_" + execution_to_fix[-15:] for col in df.columns]
            # Save
            test_preds_tabs.append(df)

# Concatenate all DataFrames into a single DataFrame
df_test_preds_tabs = pd.concat(test_preds_tabs, ignore_index=True)

In [17]:
dict_dashboard_output['Test predictions'] = df_test_preds_tabs.copy()

Now the "Execution deepdive" tab

In [18]:
# Create an empty list to store the DataFrames
exec_deep_tabs = []

# Walk through the directory
for root, dirs, files in os.walk(folder):
    for file in files:
        # Check if the file name matches the pattern
        if file.startswith("Dashboard output") and file.endswith(".xlsx"):
            # Construct the full file path
            file_path = os.path.join(root, file)
            # Read the CSV file and append to the list
            df = pd.read_excel(file_path, sheet_name="Execution deepdive")
            # Save
            exec_deep_tabs.append(df)

# Concatenate all DataFrames into a single DataFrame
df_exec_deep_tabs = pd.concat(exec_deep_tabs, ignore_index=True)
df_exec_deep_tabs['Execution name'] = execution_to_fix[-15:]

In [19]:
dict_dashboard_output['Execution deepdive'] = df_exec_deep_tabs.copy()

Lastly, the file will be exported

In [20]:
def save_dfs_to_excel(df_dict, file_name):
    """
    Saves a dictionary of pandas DataFrames to an Excel file with each DataFrame
    in a separate sheet.

    Parameters:
    df_dict (dict): A dictionary where keys are sheet names and values are DataFrames.
    file_name (str): The name of the output Excel file (should end with .xlsx).
    """
    with pd.ExcelWriter(file_name) as writer:
        for sheet_name, df in df_dict.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)

In [21]:
save_dfs_to_excel(dict_dashboard_output, "./05_outputs_modeling/output_modeling_" + execution_to_fix[-15:] + "/Dashboard output_" + execution_to_fix[-15:] + ".xlsx")